# 02 — Preprocessing Pipeline

Runs the full preprocessing pipeline for both HHAR and PAMAP2 datasets:
1. Resample to 50 Hz
2. Per-channel z-score normalisation
3. Overlapping sliding window segmentation
4. Save processed arrays to `data/processed/`

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.config import WINDOW_SIZE, OVERLAP, TARGET_SAMPLING_RATE, PROCESSED_DIR

sns.set_theme(style='whitegrid')
print(f'Window size: {WINDOW_SIZE} samples | Overlap: {OVERLAP*100:.0f}% | Target SR: {TARGET_SAMPLING_RATE} Hz')

## Step 1: Run Preprocessing

In [ ]:
from src.preprocessing import preprocess_pamap2, preprocess_hhar

# ── PAMAP2 ──
try:
    X_p2, y_p2, subj_p2 = preprocess_pamap2()
    print(f'\nPAMAP2 — Windows: {X_p2.shape}, Labels: {y_p2.shape}, Subjects: {np.unique(subj_p2)}')
except FileNotFoundError as e:
    print(f'[SKIP] {e}')

In [ ]:
# ── HHAR ──
try:
    X_hh, y_hh, subj_hh = preprocess_hhar()
    print(f'\nHHAR — Windows: {X_hh.shape}, Labels: {y_hh.shape}, Subjects: {np.unique(subj_hh)}')
except FileNotFoundError as e:
    print(f'[SKIP] {e}')

## Step 2: Verify Processed Data

In [ ]:
processed = list(Path(PROCESSED_DIR).glob('*.npy'))
print(f'Processed files ({len(processed)}):')
for f in sorted(processed):
    arr = np.load(f, allow_pickle=True)
    print(f'  {f.name:40s}  shape={arr.shape}')

## Step 3: Class & Subject Distribution

In [ ]:
try:
    X_p2 = np.load(f'{PROCESSED_DIR}/pamap2_X.npy')
    y_p2 = np.load(f'{PROCESSED_DIR}/pamap2_y.npy')
    subj_p2 = np.load(f'{PROCESSED_DIR}/pamap2_subjects.npy')

    from src.config import PAMAP2_ACTIVITIES
    activity_names = {i: name for i, name in enumerate(sorted(PAMAP2_ACTIVITIES.values()))}

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    classes, counts = np.unique(y_p2, return_counts=True)
    labels = [activity_names.get(c, str(c)) for c in classes]
    axes[0].bar(labels, counts, color='steelblue')
    axes[0].set_title('PAMAP2 — Windows per Activity')
    axes[0].tick_params(axis='x', rotation=45)

    subjs, s_counts = np.unique(subj_p2, return_counts=True)
    axes[1].bar([str(s) for s in subjs], s_counts, color='coral')
    axes[1].set_title('PAMAP2 — Windows per Subject')
    axes[1].set_xlabel('Subject ID')

    plt.tight_layout()
    plt.show()
except FileNotFoundError:
    print('Run preprocessing first.')

## Step 4: Graph Node Features Preview

In [ ]:
try:
    from src.graph_construction import (
        window_to_node_features_pamap2,
        build_pamap2_adj,
    )

    adj = build_pamap2_adj()
    print('PAMAP2 adjacency matrix (3 nodes: wrist, chest, ankle):')
    print(adj.numpy().round(3))

    node_feats = window_to_node_features_pamap2(X_p2[0])
    print(f'\nNode feature shape for one window: {node_feats.shape}')
    print('(n_nodes, feat_per_node)')
except Exception as e:
    print(f'Skipped: {e}')